# F1 - Definición del proyecto y entorno reproducible

**Proyecto:** Posicionamiento ideológico y cohesión legislativa en la tramitación del proyecto de ley sobre protección de datos personales en Chile
**Grupo 1:** Yerko Gallardo, Sebastián Rojas y José Ignacio Rodríguez  
**Curso:** MCDI500 - Programación para la Ciencia de Datos  
**Repositorio:** [grupo_1_programacion_ciencia_datos](https://github.com/joserodriguezc/grupo_1_programacion_ciencia_datos)  
**Identificador técnico del proyecto legislativo:** Boletín 11092-07

---

## Propósito del notebook

Este notebook materializa exclusivamente la **Fase 1: definición y orientación de la situación**. Integra la formulación del problema, los objetivos, el alcance y las decisiones iniciales de reproducibilidad, además de código base para:

1. centralizar la configuración mediante una clase simple;
2. registrar el entorno de ejecución;
3. inspeccionar la vinculación con Git y GitHub;
4. verificar la arquitectura esperada del repositorio;
5. comprobar la coherencia interna de la planificación.

**Límite de fase:** aquí no se consultan servicios web, no se descargan votaciones y no se transforman datos. Esas operaciones pertenecen a F2.

## 1. Contexto y problemática

La tramitación del proyecto de ley sobre protección de datos personales ha generado múltiples votaciones nominales en la Cámara de Diputadas y Diputados de Chile. Estas votaciones constituyen evidencia observable sobre el comportamiento legislativo de cada representante y sobre los patrones de acuerdo o dispersión existentes dentro de los partidos políticos.

El problema analítico consiste en determinar **qué patrones de posicionamiento relativo y cohesión legislativa se observan entre diputados y partidos políticos a partir de todas las votaciones asociadas al proyecto de ley**. El boletín 11092-07 se utiliza únicamente como identificador técnico para localizar y relacionar los registros oficiales; el objeto sustantivo es el proyecto de ley sobre protección de datos personales.

### Preguntas de análisis

1. ¿Qué posiciones relativas presentan los diputados según sus patrones de votación en esta agenda legislativa?
2. ¿Qué grado de cohesión o dispersión exhiben los partidos políticos?
3. ¿Cómo cambian los resultados al considerar participación, abstenciones, ausencias, dispensas y votaciones sin variación?
4. ¿Qué tan estables e interpretables son los resultados obtenidos mediante B-Call y especificaciones complementarias?

> **Cautela de validez:** los votos de un único proyecto permiten estimar una posición relativa respecto de esa agenda. No deben interpretarse automáticamente como una medición completa de la ideología general izquierda-derecha sin validación externa adicional.

## 2. Objetivos

### Objetivo general

Analizar el posicionamiento relativo y la cohesión legislativa de los diputados y partidos políticos de la Cámara de Diputadas y Diputados de Chile mediante todas las votaciones nominales asociadas al proyecto de ley sobre protección de datos personales, utilizando B-Call y procedimientos reproducibles de obtención, procesamiento, validación, modelamiento y comunicación de datos.

### Objetivos específicos

1. Obtener el catálogo completo y los detalles nominales de todas las votaciones asociadas al proyecto, sin restringirlas a un año determinado.
2. Identificar automáticamente los períodos legislativos correspondientes y recuperar los diputados y militancias vigentes para cada votación.
3. Explorar, limpiar, normalizar e integrar los datos, diferenciando votos, abstenciones, ausencias y dispensas.
4. Construir una tabla longitudinal y una matriz diputado por votación compatibles con los requerimientos de B-Call.
5. Estimar y contrastar posiciones relativas y cohesión a nivel de diputado, evaluando su agregación e interpretación a nivel de partido.
6. Examinar la sensibilidad, estabilidad y limitaciones del método seleccionado.
7. Comunicar resultados mediante visualizaciones, documentación científica y artefactos reproducibles.

## 3. Alcance, exclusiones y supuestos iniciales

| Dimensión | Definición inicial |
|---|---|
| Unidad de observación | Voto emitido o condición registrada para un diputado en una votación nominal. |
| Universo | Todas las votaciones asociadas al proyecto de ley identificado mediante el boletín 11092-07. |
| Cobertura temporal | Determinada automáticamente por las fechas del catálogo; no se filtra por 2023. |
| Fuente primaria | Servicios de datos abiertos de la Cámara de Diputadas y Diputados de Chile. |
| Nivel individual | Diputado por votación. |
| Nivel colectivo | Partido o militancia vigente a la fecha de cada votación. |
| Resultado esperado | Posición relativa, dispersión/cohesión, participación, diagnósticos y limitaciones. |
| Fuera de alcance en F1 | Descarga, limpieza, integración, estimación de B-Call y producción de resultados sustantivos. |

**Supuestos iniciales:**

- los identificadores de votación permiten recuperar un detalle nominal único;
- las fechas permiten asignar períodos legislativos y militancias con reglas temporales explícitas;
- las categorías de voto requerirán normalización antes del modelamiento;
- la ausencia o dispensa no equivale a abstención ni a una posición política observada;
- las votaciones unánimes o sin varianza deberán identificarse antes de construir la matriz analítica;
- cualquier inferencia a nivel de partido deberá distinguir entre posición central, dispersión interna y número efectivo de observaciones.

## 4. Configuración central del proyecto

La configuración se representa mediante una `dataclass` inmutable. Esta decisión aporta modularidad, evita valores duplicados entre notebooks y deja preparada una interfaz que podrá trasladarse posteriormente a un módulo de `src/`.

In [19]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import platform
import re
import subprocess
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 140)

print("Dependencias base importadas correctamente.")

Dependencias base importadas correctamente.


In [20]:
@dataclass(frozen=True, slots=True)
class ProjectConfig:
    '''Configuración mínima y validable del proyecto legislativo.'''

    project_name: str
    bulletin_id: str
    repository_url: str
    data_source_url: str
    python_required: str = "3.12"
    package_manager: str = "uv"
    phases: tuple[str, ...] = ("F1", "F2", "F3", "F4")

    def validate(self) -> None:
        '''Comprueba reglas estructurales sin realizar conexiones externas.'''
        if not re.fullmatch(r"\d{4,6}-\d{2}", self.bulletin_id):
            raise ValueError("El identificador del boletín no cumple el formato esperado.")
        if not self.repository_url.startswith("https://github.com/"):
            raise ValueError("La URL del repositorio debe apuntar a GitHub mediante HTTPS.")
        if self.phases != ("F1", "F2", "F3", "F4"):
            raise ValueError("La secuencia de fases debe conservar el orden F1-F4.")

    def to_record(self) -> dict[str, str]:
        '''Entrega una representación tabular legible de la configuración.'''
        record = asdict(self)
        record["phases"] = " -> ".join(self.phases)
        return record


config = ProjectConfig(
    project_name="Análisis de las votaciones del proyecto de ley sobre protección de datos personales",
    bulletin_id="11092-07",
    repository_url="https://github.com/joserodriguezc/grupo_1_programacion_ciencia_datos",
    data_source_url="https://www.camara.cl/transparencia/datosAbiertos.aspx",
)
config.validate()

pd.Series(config.to_record(), name="Valor").rename_axis("Parámetro").to_frame()

,Valor
Parámetro,
project_name,Análisis de las votaciones del proyecto de ley sobre protección de datos personales
bulletin_id,11092-07
repository_url,https://github.com/joserodriguezc/grupo_1_programacion_ciencia_datos
data_source_url,https://www.camara.cl/transparencia/datosAbiertos.aspx
python_required,3.12
package_manager,uv
phases,F1 -> F2 -> F3 -> F4


## 5. Registro reproducible del entorno

Las siguientes funciones recopilan evidencia técnica sin instalar paquetes ni modificar archivos. El resultado permite verificar la versión de Python, las principales librerías científicas y la disponibilidad de Git. Las versiones concretas corresponden al ambiente donde se ejecute por última vez el notebook.

In [21]:
def package_version(distribution: str) -> str:
    '''Retorna la versión instalada o un estado explícito si no está disponible.'''
    try:
        return version(distribution)
    except PackageNotFoundError:
        return "no instalado"


def run_command(command: list[str], cwd: Path | None = None) -> str:
    '''Ejecuta una consulta local y transforma los fallos esperables en evidencia legible.'''
    try:
        completed = subprocess.run(
            command,
            cwd=cwd,
            check=False,
            capture_output=True,
            text=True,
            timeout=10,
        )
    except (FileNotFoundError, subprocess.TimeoutExpired) as error:
        return f"no disponible ({type(error).__name__})"

    output = completed.stdout.strip() or completed.stderr.strip()
    return output if completed.returncode == 0 else f"no disponible ({output or 'sin detalle'})"


def collect_environment() -> pd.DataFrame:
    '''Construye una tabla pequeña y estable con evidencia del entorno activo.'''
    records = [
        ("Fecha de ejecución (UTC)", datetime.now(timezone.utc).isoformat(timespec="seconds")),
        ("Python", platform.python_version()),
        ("Ejecutable", sys.executable),
        ("Sistema operativo", platform.platform()),
        ("pandas", pd.__version__),
        ("NumPy", np.__version__),
        ("Jupyter Core", package_version("jupyter-core")),
        ("IPython", package_version("ipython")),
        ("uv", run_command(["uv", "--version"])),
        ("Git", run_command(["git", "--version"])),
    ]
    return pd.DataFrame(records, columns=["Componente", "Versión o evidencia"])


environment_report = collect_environment()
environment_report

,Componente,Versión o evidencia
0,Fecha de ejecución (UTC),2026-09-12T05:16:07+00:00
1,Python,3.12.13
2,Ejecutable,c:\Users\joign\OneDrive\Desktop\Magister en Ciencia de Datos e IA\1. Programación para la cienci...
3,Sistema operativo,Windows-11-10.0.26200-SP0
4,pandas,3.0.5
5,NumPy,2.5.3
6,Jupyter Core,5.9.1
7,IPython,9.17.1
8,uv,uv 0.11.11 (ed7b06001 2026-05-06 x86_64-pc-windows-msvc)
9,Git,git version 2.55.0.windows.5


## 6. Vinculación con Git/GitHub y arquitectura esperada

La inspección es deliberadamente tolerante: si el notebook se abre fuera del repositorio, informa esa condición sin interrumpir la ejecución. Una vez ubicado en la carpeta `F1/` del repositorio y ejecutado localmente, la salida debe mostrar la rama, el último commit y el remoto configurado.

In [22]:
def find_repository_root(start: Path | None = None) -> Path:
    """Busca la carpeta .git recorriendo todos los directorios superiores."""
    current = (start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (candidate / ".git").exists():
            return candidate

    raise RuntimeError(
        "No se encontró un repositorio Git desde el directorio actual."
    )


def collect_git_evidence(root: Path) -> pd.DataFrame:
    '''Recopila evidencia de trazabilidad sin alterar el historial.'''
    git_detected = (root / ".git").exists()
    if not git_detected:
        instruction = "verificar al ejecutar esta copia dentro del repositorio"
        dynamic_values = {
            "Rama activa": instruction,
            "Último commit": instruction,
            "Remoto origin": instruction,
            "Estado breve": instruction,
        }
    else:
        dynamic_values = {
            "Rama activa": run_command(["git", "branch", "--show-current"], cwd=root),
            "Último commit": run_command(["git", "log", "-1", "--pretty=%h | %s"], cwd=root),
            "Remoto origin": run_command(["git", "remote", "get-url", "origin"], cwd=root),
            "Estado breve": run_command(["git", "status", "--short"], cwd=root) or "sin cambios",
        }
    values = {
        "Raíz detectada": str(root),
        "Repositorio objetivo": config.repository_url,
        "Contexto Git detectado": "sí" if git_detected else "no; ejecutar nuevamente desde la raíz del repositorio",
        **dynamic_values,
    }
    return pd.Series(values, name="Evidencia").rename_axis("Control").to_frame()


repository_root = find_repository_root()
git_report = collect_git_evidence(repository_root)
git_report

,Evidencia
Control,
Raíz detectada,C:\Users\joign\OneDrive\Desktop\Magister en Ciencia de Datos e IA\1. Programación para la cienci...
Repositorio objetivo,https://github.com/joserodriguezc/grupo_1_programacion_ciencia_datos
Contexto Git detectado,sí
Rama activa,inicializar-entorno-reproducible
Último commit,2160ea1 | chore: inicializar entorno reproducible
Remoto origin,https://github.com/joserodriguezc/grupo_1_programacion_ciencia_datos.git
Estado breve,M README.md\n?? .vscode/\n?? F1/docs/f1_s01_evaluacion_entregable.docx\n?? F1/notebooks/


In [28]:
EXPECTED_STRUCTURE = {
    "README.md": "Descripción, instalación, ejecución y alcance",
    "pyproject.toml": "Dependencias y configuración del proyecto",
    "uv.lock": "Resolución exacta y reproducible de dependencias",
    ".python-version": "Versión objetivo de Python",
    ".gitignore": "Exclusión de ambiente, caché y datos no versionados",
    "F1/": "Definición y configuración inicial",
    "F2/": "Obtención, exploración, procesamiento y validación",
    "F2/src/": "Funciones y clases reutilizables",
    "F2/data/raw/": "Respuestas originales e inmutables",
    "F2/data/interim/": "Resultados intermedios",
    "F2/data/processed/": "Tablas analíticas validadas",
    "F2/data/manifests/": "Inventarios estructurados de cada extracción",
    "F2/tests/": "Pruebas automatizadas",
    "F2/reports/figures/": "Visualizaciones versionables",
    "F2/reports/tables/": "Tablas de resultados",
    "F2/docs/": "Decisiones, referencias y documentación complementaria",
}


def structure_report(root: Path, expected: dict[str, str]) -> pd.DataFrame:
    '''Compara la estructura esperada con el contexto de ejecución actual.'''
    rows: list[dict[str, str]] = []
    repository_context = (root / ".git").exists() or (root / "pyproject.toml").exists()
    for relative_path, purpose in expected.items():
        clean_path = relative_path.rstrip("/")
        exists = (root / clean_path).exists()
        if not repository_context:
            status = "verificar al ejecutar dentro del repositorio"
        else:
            status = "presente" if exists else "pendiente/no detectada"
        rows.append(
            {
                "Ruta": relative_path,
                "Finalidad": purpose,
                "Estado actual": status,
            }
        )
    return pd.DataFrame(rows)


repository_structure = structure_report(repository_root, EXPECTED_STRUCTURE)
repository_structure

,Ruta,Finalidad,Estado actual
0,README.md,"Descripción, instalación, ejecución y alcance",presente
1,pyproject.toml,Dependencias y configuración del proyecto,presente
2,uv.lock,Resolución exacta y reproducible de dependencias,presente
3,.python-version,Versión objetivo de Python,presente
4,.gitignore,"Exclusión de ambiente, caché y datos no versionados",presente
5,F1/,Definición y configuración inicial,presente
6,F2/,"Obtención, exploración, procesamiento y validación",presente
7,F2/src/,Funciones y clases reutilizables,presente
8,F2/data/raw/,Respuestas originales e inmutables,presente
9,F2/data/interim/,Resultados intermedios,presente


### Criterio de versionamiento

- `main` se mantiene protegida y recibe cambios mediante pull requests.
- Cada tarea se desarrolla en una rama breve (`docs/`, `chore/`, `feature/`, `analysis/`, `test/`).
- Los commits deben ser pequeños, descriptivos y vinculables con un entregable.
- Los notebooks se versionan con celdas ejecutadas en orden y sin errores; no deben contener credenciales ni datos personales.
- Los XML crudos y otros archivos voluminosos se controlan mediante reglas de `.gitignore`, manifiestos y hashes, según el tamaño y la política acordada.

**Commit de F1 esperado:** `docs: crear notebook de definición del proyecto`.

**Comando de ejecución reproducible desde la raíz del repositorio:**

```bash
uv sync
uv run jupyter nbconvert --to notebook --execute --inplace "F1/F1_Definición.ipynb"
```

## 7. Correspondencia con el mapa técnico

La tabla se genera como dato estructurado para asegurar que la secuencia del proyecto no dependa solo de una imagen. Las etapas 1 y 2 se consideran implementadas en F1; las restantes quedan explícitamente proyectadas.

In [25]:
PHASE_PLAN = [
    (1, "F1", "Definir y documentar la problemática", "F1_Definición.ipynb", "Implementada en este notebook"),
    (2, "F1", "Configurar entorno y repositorio", "F1_Definición.ipynb + archivos raíz", "Implementación inicial verificable"),
    (3, "F2", "Catalogar todas las votaciones", "F2_01_Obtencion.ipynb / script 01", "Proyectada"),
    (4, "F2", "Resolver períodos, diputados y militancias", "F2_01_Obtencion.ipynb / scripts 02-03", "Proyectada"),
    (5, "F2", "Extraer detalles nominales", "F2_01_Obtencion.ipynb / script 04", "Proyectada"),
    (6, "F2", "Realizar análisis exploratorio", "F2_02_Exploracion.ipynb", "Proyectada"),
    (7, "F2", "Procesar y normalizar", "F2_03_Procesamiento_Validacion.ipynb / script 05", "Proyectada"),
    (8, "F2", "Integrar temporalmente las fuentes", "F2_03_Procesamiento_Validacion.ipynb / script 06", "Proyectada"),
    (9, "F2", "Validar y construir matrices", "F2_03_Procesamiento_Validacion.ipynb / script 07", "Proyectada"),
    (10, "F3", "Experimentar B-Call y alternativas", "F3_01_Experimentacion.ipynb", "Proyectada"),
    (11, "F3", "Seleccionar método y especificación", "F3_02_Seleccion_Metodo.ipynb", "Proyectada"),
    (12, "F4", "Visualizar resultados", "F4_01_Visualizacion.ipynb", "Proyectada"),
    (13, "F4", "Comunicar resultados", "F4_02_Comunicacion.ipynb", "Proyectada"),
]

phase_plan = pd.DataFrame(
    PHASE_PLAN,
    columns=["N.º", "Fase", "Etapa", "Artefacto principal", "Estado al cierre de F1"],
)
phase_plan

,N.º,Fase,Etapa,Artefacto principal,Estado al cierre de F1
0,1,F1,Definir y documentar la problemática,F1_Definición.ipynb,Implementada en este notebook
1,2,F1,Configurar entorno y repositorio,F1_Definición.ipynb + archivos raíz,Implementación inicial verificable
2,3,F2,Catalogar todas las votaciones,F2_01_Obtencion.ipynb / script 01,Proyectada
3,4,F2,"Resolver períodos, diputados y militancias",F2_01_Obtencion.ipynb / scripts 02-03,Proyectada
4,5,F2,Extraer detalles nominales,F2_01_Obtencion.ipynb / script 04,Proyectada
5,6,F2,Realizar análisis exploratorio,F2_02_Exploracion.ipynb,Proyectada
6,7,F2,Procesar y normalizar,F2_03_Procesamiento_Validacion.ipynb / script 05,Proyectada
7,8,F2,Integrar temporalmente las fuentes,F2_03_Procesamiento_Validacion.ipynb / script 06,Proyectada
8,9,F2,Validar y construir matrices,F2_03_Procesamiento_Validacion.ipynb / script 07,Proyectada
9,10,F3,Experimentar B-Call y alternativas,F3_01_Experimentacion.ipynb,Proyectada


## 8. Decisiones técnicas iniciales y trazabilidad

| Decisión | Fundamento | Consecuencia verificable |
|---|---|---|
| Considerar todas las votaciones | Evita sesgo por selección temporal y respeta el alcance sustantivo del proyecto. | F2 obtiene el catálogo completo sin filtrar por año. |
| Usar el boletín como identificador técnico | Separa el objeto sustantivo de su clave de consulta. | El parámetro se centraliza en `ProjectConfig`. |
| Conservar XML y CSV | El XML preserva la respuesta original y el CSV facilita el análisis tabular. | `data/raw/` y manifiestos mantienen procedencia y hashes. |
| Resolver militancia temporalmente | La afiliación política puede cambiar durante la cobertura del proyecto. | La unión utiliza fecha de votación e intervalos de vigencia. |
| Diferenciar no participación y abstención | No representan la misma conducta legislativa. | F2 conserva categorías originales y documenta su codificación. |
| Separar notebooks y scripts | Los notebooks narran y orquestan; los módulos encapsulan lógica reutilizable. | F2 01 llama scripts 01-04 y F2 03 llama scripts 05-07. |
| Validar antes de modelar | B-Call requiere una matriz coherente y con variación informativa. | La matriz se libera solo después de controles y conciliación. |
| Proteger `main` mediante PR | Facilita revisión grupal y trazabilidad individual. | Cada integrante aporta commits identificables en ramas. |

## 9. Validaciones automáticas de Fase 1

Estas pruebas ligeras comprueban la coherencia de la configuración y del plan. No sustituyen las pruebas de datos de F2.

In [26]:
def run_f1_checks(configuration: ProjectConfig, plan: pd.DataFrame) -> pd.DataFrame:
    '''Ejecuta comprobaciones deterministas propias del alcance de F1.'''
    checks = {
        "Configuración válida": lambda: configuration.validate() is None,
        "Boletín con formato esperado": lambda: bool(re.fullmatch(r"\d{4,6}-\d{2}", configuration.bulletin_id)),
        "Repositorio GitHub declarado": lambda: configuration.repository_url.startswith("https://github.com/"),
        "Cuatro fases ordenadas": lambda: configuration.phases == ("F1", "F2", "F3", "F4"),
        "Trece etapas planificadas": lambda: len(plan) == 13,
        "Numeración única y consecutiva": lambda: plan["N.º"].tolist() == list(range(1, 14)),
        "Solo F1 figura implementada": lambda: set(plan.loc[plan["Estado al cierre de F1"].str.contains("Implement"), "Fase"]) == {"F1"},
        "Obtención sin filtro anual": lambda: "todas" in plan.loc[plan["N.º"] == 3, "Etapa"].iloc[0].lower(),
    }

    results: list[dict[str, str]] = []
    for name, check in checks.items():
        try:
            passed = bool(check())
            detail = "cumple" if passed else "no cumple"
        except Exception as error:  # La excepción se informa para facilitar diagnóstico.
            passed = False
            detail = f"error: {type(error).__name__}: {error}"
        results.append({"Control": name, "Resultado": passed, "Detalle": detail})

    report = pd.DataFrame(results)
    if not report["Resultado"].all():
        failed = report.loc[~report["Resultado"], "Control"].tolist()
        raise AssertionError(f"Validaciones F1 fallidas: {failed}")
    return report


f1_validation_report = run_f1_checks(config, phase_plan)
f1_validation_report

,Control,Resultado,Detalle
0,Configuración válida,True,cumple
1,Boletín con formato esperado,True,cumple
2,Repositorio GitHub declarado,True,cumple
3,Cuatro fases ordenadas,True,cumple
4,Trece etapas planificadas,True,cumple
5,Numeración única y consecutiva,True,cumple
6,Solo F1 figura implementada,True,cumple
7,Obtención sin filtro anual,True,cumple


## 10. Resultado de la implementación inicial

El notebook deja implementados los componentes propios de F1:

- problemática, preguntas, objetivos, alcance y supuestos;
- configuración central mediante una clase inmutable;
- funciones reutilizables para registrar entorno, Git y estructura;
- correspondencia estructurada con las 13 etapas del mapa técnico;
- decisiones iniciales de reproducibilidad y control de versiones;
- validaciones automáticas ejecutables sin conexión a internet.

La extracción de datos queda deliberadamente reservada para `F2_01_Obtencion.ipynb`. Esta separación previene que el notebook de definición mezcle formulación del problema con operaciones de ingeniería de datos.

In [27]:
summary = pd.DataFrame(
    [
        ("Configuración del proyecto", "OK"),
        ("Modularidad mediante clase y funciones", "OK"),
        ("Registro del entorno", "OK"),
        ("Vinculación declarada con GitHub", "OK"),
        ("Plan F1-F4 estructurado", "OK"),
        ("Validaciones automáticas", "OK"),
        ("Consultas a servicios web en F1", "No ejecutadas (fuera de alcance)"),
    ],
    columns=["Componente", "Estado"],
)

assert f1_validation_report["Resultado"].all()
print("F1 ejecutada de inicio a fin sin errores.")
summary

F1 ejecutada de inicio a fin sin errores.


,Componente,Estado
0,Configuración del proyecto,OK
1,Modularidad mediante clase y funciones,OK
2,Registro del entorno,OK
3,Vinculación declarada con GitHub,OK
4,Plan F1-F4 estructurado,OK
5,Validaciones automáticas,OK
6,Consultas a servicios web en F1,No ejecutadas (fuera de alcance)


## Referencias

Cámara de Diputadas y Diputados de Chile. (s. f.). *Datos abiertos*. https://www.camara.cl/transparencia/datosAbiertos.aspx

GitHub. (s. f.). *About pull requests*. https://docs.github.com/en/pull-requests/collaborating-with-pull-requests/proposing-changes-to-your-work-with-pull-requests/about-pull-requests

Project Jupyter. (s. f.). *Project Jupyter documentation*. https://docs.jupyter.org/

Python Software Foundation. (2026). *Python 3.12 documentation*. https://docs.python.org/3.12/

Toro-Maureira, S., Reutter, J., Valenzuela, L., Alcatruz, D., & Valenzuela, M. (2025). B-Call: Integrating ideological position and voting cohesion in legislative behavior. *Frontiers in Political Science, 7*, 1670089. https://doi.org/10.3389/fpos.2025.1670089

Astral. (s. f.). *uv documentation*. https://docs.astral.sh/uv/